# v7 Defect Detector — Fresh Training
**YOLO11m from COCO · Better data · Colour-invariant augmentation**

Changes vs. baseline:
- Added 6,806 images with light-coloured / plaster-wall surfaces
- `hsv_v 0.5→0.8`, `hsv_h 0.015→0.2` — colour & brightness invariance
- `copy_paste 0.5→0.8` — forces multi-defect combos every batch
- `erasing 0.4→0.1` — stops hiding defects during training
- Fresh COCO weights — no catastrophic-forgetting risk

**Before running:** upload `v7_dataset.zip` to your Google Drive root.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "ultralytics==8.4.41", "-q"])
print("Ultralytics installed.")


In [ ]:
import os, zipfile, pathlib

ZIP_SRC = pathlib.Path("/content/drive/MyDrive/v7_dataset.zip")
DATASET = pathlib.Path("/content/v7_dataset")

if not ZIP_SRC.exists():
    raise FileNotFoundError(
        "v7_dataset.zip not found in Google Drive root.\n"
        "Build it locally with:\n"
        "  python scripts/build_v7_dataset.py\n"
        "  cd output && zip -r v7_dataset.zip v7_dataset\n"
        "Then upload v7_dataset.zip to Google Drive."
    )

if not DATASET.exists():
    print(f"Unzipping {ZIP_SRC} …")
    with zipfile.ZipFile(ZIP_SRC) as zf:
        zf.extractall("/content/")
    print(f"Done. {sum(1 for _ in DATASET.rglob('*.jpg'))+sum(1 for _ in DATASET.rglob('*.png'))} images found.")
else:
    print("Dataset already unzipped — skipping.")

# Patch data.yaml to use Colab paths
YAML = DATASET / "data.yaml"
text = YAML.read_text()
# Replace any Windows path with the Colab path
import re
text = re.sub(r'^path:.*$', f'path: {DATASET.as_posix()}', text, flags=re.MULTILINE)
YAML.write_text(text)
print("data.yaml patched.")
print(YAML.read_text())


In [ ]:
import shutil, csv
from pathlib import Path
from ultralytics import YOLO

DATASET_YAML = Path("/content/v7_dataset/data.yaml")
OUT_DIR      = Path("/content/v7_runs")
DRIVE_OUT    = Path("/content/drive/MyDrive/AIEngGroupProj_v7_weights")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

model = YOLO("yolo11m.pt")   # downloads COCO pretrained from Ultralytics CDN

print("=" * 64)
print("v7 — Fresh YOLO11m, improved augmentation + data")
print("=" * 64)

model.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=640,
    batch=32,           # Colab A100/V100 can handle larger batch
    workers=4,
    device="0",

    # Optimiser
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=3,
    warmup_bias_lr=0.1,
    warmup_momentum=0.8,
    cos_lr=True,

    # Loss
    cls=1.5,

    # Augmentation — KEY CHANGES
    hsv_h=0.2,
    hsv_s=0.9,
    hsv_v=0.8,
    copy_paste=0.8,
    mosaic=1.0,
    scale=0.5,
    perspective=0.001,
    degrees=15.0,
    flipud=0.3,
    fliplr=0.5,
    mixup=0.1,
    erasing=0.1,

    # Config
    patience=20,
    save_period=5,
    exist_ok=True,
    project=str(OUT_DIR),
    name="v7_fresh",
)
print("Training complete.")


In [ ]:
import shutil, csv
from pathlib import Path

OUT_DIR   = Path("/content/v7_runs")
DRIVE_OUT = Path("/content/drive/MyDrive/AIEngGroupProj_v7_weights")

best_pt = OUT_DIR / "v7_fresh" / "weights" / "best.pt"
last_pt = OUT_DIR / "v7_fresh" / "weights" / "last.pt"
csv_path = OUT_DIR / "v7_fresh" / "results.csv"

target = best_pt if best_pt.exists() else last_pt
shutil.copy2(target, DRIVE_OUT / "defect_detector_v7_candidate.pt")
if csv_path.exists():
    shutil.copy2(csv_path, DRIVE_OUT / "v7_results.csv")

# Report mAP50
if csv_path.exists():
    rows = list(csv.DictReader(csv_path.open()))
    m50 = max(float(r.get("metrics/mAP50(B)", 0)) for r in rows)
    baseline = 0.694
    flag = "✓ BEATS BASELINE" if m50 > baseline else "✗ below baseline"
    print(f"Best mAP50 = {m50:.4f}  {flag}  (baseline={baseline})")

print(f"Weights saved → {DRIVE_OUT / 'defect_detector_v7_candidate.pt'}")
print("\nNext steps (run locally):")
print("  1. Download defect_detector_v7_candidate.pt from Drive")
print("  2. Copy to: AIEngGroupProj/weights/candidates/current/")
print("  3. Update inference_api.py model priority if needed")
